In [17]:
!pip install --upgrade torchao

In [18]:
import numpy as np
import pandas as pd
from diffusers import UNet2DConditionModel, AutoencoderKL, DDPMScheduler
from transformers import CLIPTokenizer, CLIPTextModel
from peft import LoraConfig, get_peft_model
import torch
from torch import optim, nn
import os
from PIL import Image
from torchvision import transforms
import torch.nn.functional as F

In [19]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = 'stable-diffusion-v1-5/stable-diffusion-v1-5'

tokenizer = CLIPTokenizer.from_pretrained(model_name, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(model_name, subfolder="text_encoder").to(device) 
noise_scheduler = DDPMScheduler.from_pretrained(model_name, subfolder="scheduler")

vae = AutoencoderKL.from_pretrained(model_name, subfolder="vae").to(device)
unet = UNet2DConditionModel.from_pretrained(model_name, subfolder="unet").to(device)
vae.requires_grad_(False)

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: stable-diffusion-v1-5/stable-diffusion-v1-5
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


AutoencoderKL(
  (encoder): Encoder(
    (conv_in): Conv2d(3, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (down_blocks): ModuleList(
      (0): DownEncoderBlock2D(
        (resnets): ModuleList(
          (0-1): 2 x ResnetBlock2D(
            (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
            (conv1): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (norm2): GroupNorm(32, 128, eps=1e-06, affine=True)
            (dropout): Dropout(p=0.0, inplace=False)
            (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
            (nonlinearity): SiLU()
          )
        )
        (downsamplers): ModuleList(
          (0): Downsample2D(
            (conv): Conv2d(128, 128, kernel_size=(3, 3), stride=(2, 2))
          )
        )
      )
      (1): DownEncoderBlock2D(
        (resnets): ModuleList(
          (0): ResnetBlock2D(
            (norm1): GroupNorm(32, 128, eps=1e-06, affine=True)
            (c

In [20]:
train_meta = pd.read_csv("/kaggle/input/competitions/flowers-generation/data/flowers102/train/meta.csv")
id2label = pd.read_csv("/kaggle/input/competitions/flowers-generation/data/id2label.csv")
idx2label = dict(zip(id2label['ID'].tolist(), id2label['Label'].tolist()))
train_meta['named'] = train_meta['Label'].apply(lambda x: idx2label[x])
train_img = '/kaggle/input/competitions/flowers-generation/data/flowers102/train/imgs'
train_meta['ID'] = train_meta['ID'].astype(str) + '.jpg'
train_meta

,ID,Label,named
0,34.jpg,0,pink primrose
1,78.jpg,0,pink primrose
2,204.jpg,0,pink primrose
3,388.jpg,1,canterbury bells
4,463.jpg,1,canterbury bells
...,...,...,...
104,6347.jpg,0,pink primrose
105,6481.jpg,1,canterbury bells
106,6495.jpg,1,canterbury bells
107,6518.jpg,2,sweet pea


In [21]:
from torch.utils.data import Dataset
from PIL import Image
import os

class FlowerDataset(Dataset):
    def __init__(
        self,
        dataframe,
        image_dir,
        tokenizer,
        image_size=512
    ):
        self.df = dataframe
        self.image_dir = image_dir
        self.tokenizer = tokenizer
        
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])  # Standard [-1, 1] norm for 3-channel RGB
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        image_path = os.path.join(self.image_dir, row["ID"])
        image = Image.open(image_path).convert("RGB")
        image = self.transform(image)
        
        prompt = f"a photo of a {row['named']}"
        tokens = self.tokenizer(
            prompt,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        return {
            "pixel_values": image,
            "input_ids": tokens.input_ids.squeeze(0),
            "attention_mask": tokens.attention_mask.squeeze(0)  # Recommended addition
        }

In [22]:
dataset = FlowerDataset(
    train_meta,
    train_img,
    tokenizer
)

In [23]:
from torch.utils.data import DataLoader
epochs = 15
loader = DataLoader(
    dataset,
    batch_size=8, 
    shuffle=True
)

In [24]:
config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "to_q",
        "to_k",
        "to_v",
        "to_out.0",
    ],
    lora_dropout=0.1,
)

unet = get_peft_model(unet, config)
unet.enable_gradient_checkpointing()
try:
    unet.enable_xformers_memory_efficient_attention()
except:
    pass

unet.print_trainable_parameters()


trainable params: 1,594,368 || all params: 861,115,332 || trainable%: 0.1852


In [26]:
from transformers import get_cosine_schedule_with_warmup

device = torch.device("cuda")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

vae.eval()
vae.requires_grad_(False)

text_encoder.eval()
text_encoder.requires_grad_(False)

unet.train()

trainable_params = [p for p in unet.parameters() if p.requires_grad]

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=1e-5,
    weight_decay=1e-2,
)

accumulation_steps = 8

num_training_steps = (epochs * len(loader) + accumulation_steps - 1) // accumulation_steps

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(10, num_training_steps // 20),
    num_training_steps=num_training_steps,
)

scaler = torch.amp.GradScaler("cuda")

for epoch in range(epochs):
    losses = []
    optimizer.zero_grad(set_to_none=True)

    for i, batch in enumerate(tqdm(loader)):
        images = batch["pixel_values"].to(device, non_blocking=True)
        input_ids = batch["input_ids"].to(device, non_blocking=True)

        with torch.no_grad():
            latents = vae.encode(images).latent_dist.sample()
            latents = latents * vae.config.scaling_factor
            text_embeddings = text_encoder(input_ids).last_hidden_state

        noise = torch.randn_like(latents)

        timesteps = torch.randint(
            0,
            noise_scheduler.config.num_train_timesteps,
            (latents.size(0),),
            device=device,
            dtype=torch.long,
        )

        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        with torch.amp.autocast("cuda", dtype=torch.float16):
            noise_pred = unet(
                noisy_latents,
                timesteps,
                encoder_hidden_states=text_embeddings,
            ).sample

            loss = F.mse_loss(noise_pred.float(), noise.float())
            loss = loss / accumulation_steps

        scaler.scale(loss).backward()
        losses.append(loss.item() * accumulation_steps)

        if (i + 1) % accumulation_steps == 0 or (i + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

    print(f"Epoch {epoch}: {sum(losses)/len(losses):.4f}")

100%|██████████| 14/14 [00:59<00:00,  4.26s/it]


Epoch 0: 0.1363


100%|██████████| 14/14 [00:57<00:00,  4.12s/it]


Epoch 1: 0.1395


100%|██████████| 14/14 [00:58<00:00,  4.16s/it]


...

Epoch 7: 0.1810


100%|██████████| 14/14 [00:58<00:00,  4.17s/it]


Epoch 8: 0.1871


 14%|█▍        | 2/14 [00:09<00:58,  4.86s/it]


KeyboardInterrupt: 

In [29]:
test = pd.read_csv("/kaggle/input/competitions/flowers-generation/data/flowers102/test/test_prompts.csv")
from diffusers import StableDiffusionPipeline

pipe = StableDiffusionPipeline.from_pretrained(
    model_name,
    vae=vae,
    text_encoder=text_encoder,
    tokenizer=tokenizer,
    unet=unet,
    scheduler=noise_scheduler,
    safety_checker=None,
    feature_extractor=None,
    requires_safety_checker=False,
    torch_dtype=torch.float16,
).to(device)

pipe.set_progress_bar_config(disable=True)

generator = torch.Generator(device=device).manual_seed(42)

rows = []

for image_id, prompt in tqdm(zip(test["ID"], test["Prompts"]), total=len(test)):
    image = pipe(
        prompt=prompt,
        num_inference_steps=50,
        guidance_scale=7.5,
        height=512,
        width=512,
        generator=generator,
    ).images[0]

    image = np.array(image, dtype=np.uint8)

    for channel_name, channel_idx in zip(["R", "G", "B"], [0, 1, 2]):
        rows.append({
            "row_id": f"{image_id}_{channel_name}",
            "image_id": image_id,
            "color": channel_name,
            "values": " ".join(map(str, image[:, :, channel_idx].reshape(-1)))
        })

submission = pd.DataFrame(rows)
submission.to_csv("submission.csv", index=False)

print(submission.head())
print("Saved submission.csv")

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Expected types for unet: (<class 'diffusers.models.unets.unet_2d_condition.UNet2DConditionModel'>,), got <class 'peft.peft_model.PeftModel'>.
100%|██████████| 14/14 [06:05<00:00, 26.11s/it]


  row_id  image_id color                                             values
0   70_R        70     R  129 129 129 129 129 129 128 129 127 128 129 12...
1   70_G        70     G  114 113 112 114 112 113 112 113 111 112 112 11...
2   70_B        70     B  123 124 129 130 129 128 127 128 125 127 127 12...
3   95_R        95     R  2 0 5 3 5 4 4 4 4 4 4 5 4 4 4 5 5 5 5 5 5 4 4 ...
4   95_G        95     G  1 0 2 4 3 2 3 3 3 3 3 3 4 3 3 3 3 4 3 4 3 3 2 ...
Saved submission.csv
